# 🎙️ SLAM-LLM — ASR LibriSpeech (Google Colab)

**Drive Kullanımı — Minimum:**
- ✅ LibriSpeech zaten Drive'da → sadece okunur, kopyalanmaz
- ✅ Vicuna-7B → HuggingFace cache'de kalır (Drive'a yazılmaz)
- ✅ Encoder checkpoint'leri → Colab'da `/content/models` altında kalır
- ✅ **Drive'a sadece** eğitim çıktısı (projector checkpoint ~20MB) kaydedilir

**İçerik:**
- Bölüm 1 — Kurulum
- Bölüm 2 — Veri Hazırlama (Drive'daki mevcut LibriSpeech'ten JSONL üret)
- **Bölüm 3 — Hızlı Test** ⚡ (Whisper, batch=1, 100 step)
- **Bölüm 4 — Konfig A** 🏋️ (WavLM-Large → WER: 2.28 / 4.78)
- **Bölüm 5 — Konfig B** 🏆 (HuBERT XtraLarge → WER: 1.84 / 3.39)

> 📄 Paper: [SLAM-ASR (arXiv:2402.08846)](https://arxiv.org/abs/2402.08846)

## 📦 Bölüm 1: Ortam Kurulumu

In [ ]:
# ─── GPU Kontrolü ────────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

: 

In [ ]:
# ─── Google Drive Bağlantısı ─────────────────────────────────────────────────
# Sadece LibriSpeech'i OKUMAK ve checkpoint (~20MB) YAZMAK için kullanılıyor.
from google.colab import drive
drive.mount('/content/drive')

import os

# Drive'daki mevcut LibriSpeech yolu
LIBRISPEECH_DRIVE = '/content/drive/MyDrive/datasets/speech/LibriSpeech'

# Checkpoint çıktısı için küçük bir klasör
CKPT_OUT_DIR = '/content/drive/MyDrive/SLAM-LLM/outputs'
os.makedirs(CKPT_OUT_DIR, exist_ok=True)

# Drive'daki LibriSpeech'i doğrula
if os.path.exists(LIBRISPEECH_DRIVE):
    splits = os.listdir(LIBRISPEECH_DRIVE)
    print(f"✅ LibriSpeech bulundu: {LIBRISPEECH_DRIVE}")
    print(f"   Mevcut split'ler: {splits}")
else:
    print(f"⚠️  LibriSpeech bulunamadı: {LIBRISPEECH_DRIVE}")
    print("   Lütfen Drive yolunu kontrol edin.")

In [ ]:
# ─── Repo Klonlama ───────────────────────────────────────────────────────────
%cd /content
!git clone https://github.com/X-LANCE/SLAM-LLM.git
%cd /content/SLAM-LLM
!git log --oneline -3

In [ ]:
# ─── Python Bağımlılıkları ───────────────────────────────────────────────────
%cd /content/SLAM-LLM
!pip install -q -e .
!pip install -q \
    transformers==4.37.2 \
    peft==0.9.0 \
    datasets soundfile librosa \
    hydra-core omegaconf \
    deepspeed fairscale fire

# fairseq — WavLM ve HuBERT için gerekli
!pip install -q fairseq

print("\n✅ Bağımlılıklar kuruldu.")

In [ ]:
# ─── PYTHONPATH ──────────────────────────────────────────────────────────────
import os, sys
os.environ['PYTHONPATH'] = '/content/SLAM-LLM/src'
sys.path.insert(0, '/content/SLAM-LLM/src')
sys.path.insert(0, '/content/SLAM-LLM')
print("PYTHONPATH ayarlandı.")

## 📊 Bölüm 2: Veri Hazırlama

Drive'daki mevcut LibriSpeech ses dosyalarından JSONL üretilir.  
**Ses dosyaları kopyalanmaz, sadece yolları referans edilir.**  
JSONL dosyaları Colab'ın geçici belleğine (`/content/data/jsonl`) yazılır — Drive'ı doldurmaz.

In [ ]:
# ─── Drive'daki LibriSpeech Yapısını İncele ──────────────────────────────────
import os, glob

LIBRISPEECH_DRIVE = '/content/drive/MyDrive/datasets/speech/LibriSpeech'

print("Drive'daki LibriSpeech yapısı:")
for split in sorted(os.listdir(LIBRISPEECH_DRIVE)):
    split_path = os.path.join(LIBRISPEECH_DRIVE, split)
    if os.path.isdir(split_path):
        # Ses dosyası sayısını say
        flac_files = glob.glob(f'{split_path}/**/*.flac', recursive=True)
        trans_files = glob.glob(f'{split_path}/**/*.trans.txt', recursive=True)
        print(f"  {split:30s}  {len(flac_files):5d} ses   {len(trans_files):3d} trans")

In [ ]:
# ─── JSONL Üret (ses dosyaları Drive'da kalır, sadece yollar yazılır) ─────────
import os, json, glob

LIBRISPEECH_DRIVE = '/content/drive/MyDrive/datasets/speech/LibriSpeech'
JSONL_DIR = '/content/data/jsonl'   # ← Colab geçici bellek, Drive'ı doldurmaz
os.makedirs(JSONL_DIR, exist_ok=True)

def librispeech_to_jsonl(split_name, output_path, max_samples=None):
    """
    Drive'daki LibriSpeech split'ini SLAM-LLM JSONL formatına çevirir.
    Ses dosyaları KOPYALANMAZ — Drive yolları referans edilir.
    """
    split_dir = os.path.join(LIBRISPEECH_DRIVE, split_name)
    if not os.path.exists(split_dir):
        print(f"⚠️  Split bulunamadı: {split_dir}")
        return 0

    records = []
    trans_files = sorted(glob.glob(f'{split_dir}/**/*.trans.txt', recursive=True))

    for trans_file in trans_files:
        trans_dir = os.path.dirname(trans_file)
        with open(trans_file) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(' ', 1)
                if len(parts) != 2:
                    continue
                utt_id, transcript = parts
                audio_path = os.path.join(trans_dir, f'{utt_id}.flac')
                if os.path.exists(audio_path):
                    records.append({
                        'key': f'{utt_id}_ASR',
                        'source': audio_path,          # Drive yolu — kopyalanmaz
                        'target': transcript.lower()
                    })
        if max_samples and len(records) >= max_samples:
            records = records[:max_samples]
            break

    with open(output_path, 'w') as f:
        for rec in records:
            f.write(json.dumps(rec) + '\n')

    size_kb = os.path.getsize(output_path) / 1024
    print(f"✅ {output_path.split('/')[-1]}: {len(records)} örnek | {size_kb:.0f} KB")
    return len(records)


# ── Hızlı test için küçük subset (train-clean-100'den ilk 500 örnek) ──
librispeech_to_jsonl('train-clean-100', f'{JSONL_DIR}/train_mini.jsonl',    max_samples=500)

# ── Validation: dev-clean ──
librispeech_to_jsonl('dev-clean',      f'{JSONL_DIR}/dev_clean.jsonl')

# ── Full training: train-clean-360 + train-other-500 (960h) ──
# (Drive'dan okunur, kopyalanmaz — sadece yollar yazılır)
librispeech_to_jsonl('train-clean-360', f'{JSONL_DIR}/train_clean_360.jsonl')
librispeech_to_jsonl('train-other-500', f'{JSONL_DIR}/train_other_500.jsonl')

# ── 960h = clean-100 + clean-360 + other-500 birleştir ──
jsonl_960h = f'{JSONL_DIR}/train_960h.jsonl'
with open(jsonl_960h, 'w') as out:
    for part in ['train-clean-100', 'train-clean-360', 'train-other-500']:
        part_path = f'{JSONL_DIR}/{part.replace("-", "_")}.jsonl'
        # train-clean-100 zaten mini olarak üretildi, tam hâlini burada üretelim
        pass

# Temiz 960h üretimi
all_records = []
for split in ['train-clean-100', 'train-clean-360', 'train-other-500']:
    split_dir = os.path.join(LIBRISPEECH_DRIVE, split)
    if not os.path.exists(split_dir):
        print(f"  ⚠️  {split} Drive'da bulunamadı, atlanıyor.")
        continue
    for trans_file in sorted(glob.glob(f'{split_dir}/**/*.trans.txt', recursive=True)):
        trans_dir = os.path.dirname(trans_file)
        with open(trans_file) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = line.split(' ', 1)
                if len(parts) != 2: continue
                utt_id, transcript = parts
                audio_path = os.path.join(trans_dir, f'{utt_id}.flac')
                if os.path.exists(audio_path):
                    all_records.append({'key': f'{utt_id}_ASR', 'source': audio_path, 'target': transcript.lower()})

with open(jsonl_960h, 'w') as f:
    for rec in all_records:
        f.write(json.dumps(rec) + '\n')

size_mb = os.path.getsize(jsonl_960h) / 1e6
print(f"✅ train_960h.jsonl: {len(all_records)} örnek | {size_mb:.1f} MB")

# ── Test set ──
librispeech_to_jsonl('test-clean', f'{JSONL_DIR}/test_clean.jsonl')

print(f"\n📁 Tüm JSONL'ler Colab'ın /content/data/jsonl dizininde — Drive'da yer kaplamaz.")

In [ ]:
# ─── HuggingFace Token (Colab Secrets) ──────────────────────────────────────
# lmsys/vicuna-7b-v1.5 için gerekli.
#
# 🔑 Token'ınızı Colab Secrets'a ekleyin:
#   1. Sol panelde 🔑 (anahtar) ikonuna tıklayın
#   2. "Add a new secret" → Name: HF_TOKEN, Value: hf_... token'ınız
#   3. "Notebook access" toggle'ını açın
#
# Token al: https://huggingface.co/settings/tokens
# Model erişimi: https://huggingface.co/lmsys/vicuna-7b-v1.5 (accept agreement)

from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
!huggingface-cli login --token $HF_TOKEN

print("✅ HF_TOKEN Colab Secrets'tan alındı.")
# Vicuna-7B: HuggingFace cache'e indirilir (/root/.cache/huggingface/)
# Drive'a yazılmaz → Drive alanı harcanmaz
print("💡 Vicuna-7B HuggingFace cache'e indirilecek (/root/.cache) — Drive'a yazılmaz.")
print("   Session kapandığında silinir, bir sonraki session'da tekrar indirilir (~14GB, ~10 dk).")

## ⚡ Bölüm 3: Hızlı Test — Whisper-Large-v3 (T4 Uyumlu)

| Parametre | Değer |
|-----------|-------|
| Encoder | Whisper-large-v3 (HF'den otomatik) |
| LLM | vicuna-7b-v1.5 (HF cache) |
| Batch size | 1 · fp16 |
| Steps | **100** (sadece pipeline testi) |
| Drive'a yazılan | Projector checkpoint (~20MB) |

In [ ]:
# ─── HIZLI TEST ───────────────────────────────────────────────────────────────
import os, datetime
os.chdir('/content/SLAM-LLM')

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
OUT_QUICK = f'/content/drive/MyDrive/SLAM-LLM/outputs/quick_test_{ts}'  # ~20MB
os.makedirs(OUT_QUICK, exist_ok=True)

cmd_quick = f"""
export PYTHONPATH=/content/SLAM-LLM/src:$PYTHONPATH
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
export OMP_NUM_THREADS=1

python examples/asr_librispeech/finetune_asr.py \\
  --config-path conf \\
  --config-name prompt.yaml \\
  hydra.run.dir={OUT_QUICK} \\
  ++model_config.llm_name=vicuna-7b-v1.5 \\
  ++model_config.llm_path=lmsys/vicuna-7b-v1.5 \\
  ++model_config.llm_dim=4096 \\
  ++model_config.encoder_name=whisper \\
  ++model_config.encoder_projector_ds_rate=5 \\
  ++model_config.encoder_path=openai/whisper-large-v3 \\
  ++model_config.encoder_dim=1280 \\
  ++model_config.encoder_projector=linear \\
  ++dataset_config.dataset=speech_dataset \\
  ++dataset_config.train_data_path=/content/data/jsonl/train_mini.jsonl \\
  ++dataset_config.val_data_path=/content/data/jsonl/dev_clean.jsonl \\
  ++dataset_config.input_type=mel \\
  ++dataset_config.mel_size=128 \\
  ++train_config.model_name=asr \\
  ++train_config.num_epochs=1 \\
  ++train_config.freeze_encoder=true \\
  ++train_config.freeze_llm=true \\
  ++train_config.batching_strategy=custom \\
  ++train_config.warmup_steps=10 \\
  ++train_config.total_steps=100 \\
  ++train_config.lr=1e-4 \\
  ++train_config.validation_interval=50 \\
  ++train_config.batch_size_training=1 \\
  ++train_config.val_batch_size=1 \\
  ++train_config.num_workers_dataloader=2 \\
  ++train_config.use_fp16=true \\
  ++train_config.output_dir={OUT_QUICK} \\
  ++metric=acc
"""

print("⚡ Hızlı test (100 step, Whisper encoder, T4 uyumlu)")
print(f"📁 Checkpoint → {OUT_QUICK}  (~20MB)")
print("=" * 60)
!bash -c "{cmd_quick}"

: 

: 

---
## 🏋️ Bölüm 4: Konfig A — WavLM-Large + Vicuna-7B

README Performans tablosunun **1. satırı**:

| test-clean WER | test-other WER | Projector |
|:-:|:-:|---|
| **2.28** | **4.78** | Linear ~18.88M |

**Drive'a yazılan:** Sadece projector checkpoint (~20MB)  
**GPU:** A100 40GB önerilir (batch=4, fp16)

In [ ]:
# ─── [Konfig A] WavLM-Large Encoder İndir (Colab'a, Drive'a değil) ───────────
# WavLM-Large: ~1.26 GB
# Google Drive File ID: 12-cB34qCTvByWT-QtOcZaqwwO21FLSqU
import os

WAVLM_PATH = '/content/models/WavLM-Large.pt'   # Colab geçici bellek
os.makedirs('/content/models', exist_ok=True)

if not os.path.exists(WAVLM_PATH):
    print("WavLM-Large indiriliyor (~1.26 GB, Colab'a)...")
    !pip install -q gdown
    !gdown --id 12-cB34qCTvByWT-QtOcZaqwwO21FLSqU -O {WAVLM_PATH}
else:
    print(f"✅ WavLM-Large zaten mevcut: {WAVLM_PATH}")

size_gb = os.path.getsize(WAVLM_PATH) / 1e9
print(f"   Boyut: {size_gb:.2f} GB")
print("💡 Bu dosya Colab'da kalır — Drive'a yazılmaz.")

In [ ]:
# ─── [Konfig A] WavLM-Large + Vicuna-7B Eğitimi ──────────────────────────────
# README: test-clean WER=2.28, test-other WER=4.78
import os, datetime
os.chdir('/content/SLAM-LLM')

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
OUT_A = f'/content/drive/MyDrive/SLAM-LLM/outputs/wavlm_large_{ts}'  # sadece ckpt ~20MB
os.makedirs(OUT_A, exist_ok=True)

TRAIN_DATA = '/content/data/jsonl/train_960h.jsonl'    # Drive'daki seslere referans verir
VAL_DATA   = '/content/data/jsonl/dev_clean.jsonl'
WAVLM_PATH = '/content/models/WavLM-Large.pt'

cmd_a = f"""
export PYTHONPATH=/content/SLAM-LLM/src:$PYTHONPATH
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
export OMP_NUM_THREADS=1

python examples/asr_librispeech/finetune_asr.py \\
  --config-path conf \\
  --config-name prompt.yaml \\
  hydra.run.dir={OUT_A} \\
  ++model_config.llm_name=vicuna-7b-v1.5 \\
  ++model_config.llm_path=lmsys/vicuna-7b-v1.5 \\
  ++model_config.llm_dim=4096 \\
  ++model_config.encoder_name=wavlm \\
  ++model_config.normalize=true \\
  ++dataset_config.normalize=true \\
  ++model_config.encoder_projector_ds_rate=5 \\
  ++model_config.encoder_path={WAVLM_PATH} \\
  ++model_config.encoder_dim=1024 \\
  ++model_config.encoder_projector=linear \\
  ++dataset_config.dataset=speech_dataset \\
  ++dataset_config.train_data_path={TRAIN_DATA} \\
  ++dataset_config.val_data_path={VAL_DATA} \\
  ++dataset_config.input_type=raw \\
  ++train_config.model_name=asr \\
  ++train_config.num_epochs=3 \\
  ++train_config.freeze_encoder=true \\
  ++train_config.freeze_llm=true \\
  ++train_config.batching_strategy=custom \\
  ++train_config.warmup_steps=1000 \\
  ++train_config.total_steps=100000 \\
  ++train_config.lr=1e-4 \\
  ++train_config.validation_interval=1000 \\
  ++train_config.batch_size_training=4 \\
  ++train_config.val_batch_size=4 \\
  ++train_config.num_workers_dataloader=2 \\
  ++train_config.use_fp16=true \\
  ++train_config.output_dir={OUT_A} \\
  ++metric=acc
"""

print("🏋️ Konfig A: WavLM-Large + Vicuna-7B")
print(f"🎯 Hedef: test-clean WER=2.28, test-other WER=4.78")
print(f"📁 Checkpoint (Drive): {OUT_A}")
print("=" * 60)
!bash -c "{cmd_a}"

---
## 🏆 Bölüm 5: Konfig B — HuBERT XtraLarge + Vicuna-7B (En İyi)

README Performans tablosunun **2. satırı** (en iyi):

| test-clean WER | test-other WER | Projector |
|:-:|:-:|---|
| **1.84** | **3.39** | Linear ~21.50M |

**Drive'a yazılan:** Sadece projector checkpoint (~20MB)  
**GPU:** A100 40GB önerilir (batch=6, fp16)

In [ ]:
# ─── [Konfig B] HuBERT XtraLarge Encoder İndir (Colab'a, Drive'a değil) ──────
# Meta AI resmi kaynağı: ~1.35 GB
import os

HUBERT_PATH = '/content/models/hubert_xtralarge_ll60k_finetune_ls960.pt'
os.makedirs('/content/models', exist_ok=True)

if not os.path.exists(HUBERT_PATH):
    print("HuBERT XtraLarge indiriliyor (~1.35 GB, Colab'a)...")
    !wget -q \
        https://dl.fbaipublicfiles.com/hubert/hubert_xtralarge_ll60k_finetune_ls960.pt \
        -O {HUBERT_PATH}
else:
    print(f"✅ HuBERT XtraLarge zaten mevcut: {HUBERT_PATH}")

size_gb = os.path.getsize(HUBERT_PATH) / 1e9
print(f"   Boyut: {size_gb:.2f} GB")
print("💡 Bu dosya Colab'da kalır — Drive'a yazılmaz.")

In [ ]:
# ─── [Konfig B] HuBERT XtraLarge + Vicuna-7B Eğitimi ────────────────────────
# README: test-clean WER=1.84, test-other WER=3.39 (EN İYİ)
import os, datetime
os.chdir('/content/SLAM-LLM')

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
OUT_B = f'/content/drive/MyDrive/SLAM-LLM/outputs/hubert_xtralarge_{ts}'  # sadece ckpt ~21MB
os.makedirs(OUT_B, exist_ok=True)

TRAIN_DATA  = '/content/data/jsonl/train_960h.jsonl'
VAL_DATA    = '/content/data/jsonl/dev_clean.jsonl'
HUBERT_PATH = '/content/models/hubert_xtralarge_ll60k_finetune_ls960.pt'

cmd_b = f"""
export PYTHONPATH=/content/SLAM-LLM/src:$PYTHONPATH
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
export OMP_NUM_THREADS=1

python examples/asr_librispeech/finetune_asr.py \\
  --config-path conf \\
  --config-name prompt.yaml \\
  hydra.run.dir={OUT_B} \\
  ++model_config.llm_name=vicuna-7b-v1.5 \\
  ++model_config.llm_path=lmsys/vicuna-7b-v1.5 \\
  ++model_config.llm_dim=4096 \\
  ++model_config.encoder_name=hubert \\
  ++model_config.normalize=true \\
  ++dataset_config.normalize=true \\
  ++model_config.encoder_projector_ds_rate=5 \\
  ++model_config.encoder_path={HUBERT_PATH} \\
  ++model_config.encoder_dim=1280 \\
  ++model_config.encoder_type=finetune \\
  ++model_config.encoder_projector=linear \\
  ++dataset_config.dataset=speech_dataset \\
  ++dataset_config.train_data_path={TRAIN_DATA} \\
  ++dataset_config.val_data_path={VAL_DATA} \\
  ++dataset_config.input_type=raw \\
  ++train_config.model_name=asr \\
  ++train_config.num_epochs=3 \\
  ++train_config.freeze_encoder=true \\
  ++train_config.freeze_llm=true \\
  ++train_config.batching_strategy=custom \\
  ++train_config.warmup_steps=1000 \\
  ++train_config.total_steps=100000 \\
  ++train_config.lr=1e-4 \\
  ++train_config.validation_interval=2000 \\
  ++train_config.batch_size_training=6 \\
  ++train_config.val_batch_size=6 \\
  ++train_config.num_workers_dataloader=0 \\
  ++train_config.use_fp16=true \\
  ++train_config.output_dir={OUT_B} \\
  ++metric=acc
"""

print("🏆 Konfig B: HuBERT XtraLarge + Vicuna-7B (EN İYİ)")
print(f"🎯 Hedef: test-clean WER=1.84, test-other WER=3.39")
print(f"📁 Checkpoint (Drive): {OUT_B}")
print("=" * 60)
!bash -c "{cmd_b}"

---
## 📋 Drive Kullanım Özeti

| Bileşen | Boyut | Nerede? |
|---------|-------|--------|
| LibriSpeech ses | ~55 GB | ✅ Drive'da zaten var (kopyalanmaz) |
| JSONL metadata | ~10 MB | ⚡ Colab geçici (`/content`) — Drive'a yazılmaz |
| Vicuna-7B | ~14 GB | ⚡ HF cache (`/root/.cache`) — Drive'a yazılmaz |
| WavLM-Large encoder | ~1.26 GB | ⚡ Colab geçici — Drive'a yazılmaz |
| HuBERT XtraLarge encoder | ~1.35 GB | ⚡ Colab geçici — Drive'a yazılmaz |
| **Projector checkpoint** | **~20 MB** | 💾 **Drive'a yazılan tek şey** |

### ⚠️ Notlar
- Session kapandığında Vicuna + encoder'lar silinir → sonraki session'da tekrar indirilir
- Eğer encoder'ı da Drive'a kaydetmek istiyorsanız `WAVLM_PATH` / `HUBERT_PATH` değişkenini Drive yolu ile değiştirin
- `train-clean-100` / `train-clean-360` / `train-other-500` split'leri Drive'da yoksa ilgili hücre uyarı verir ve atlar